> **Guía de lectura ampliada.** Cada análisis tiene una explicación de **por qué se hace, qué se hace y cómo se hace**, seguida de su **conclusión** junto a los resultados.
> Las conclusiones numéricas interpretan la ejecución guardada y revisada el **7 de octubre de 2026**. Si cambian los datos, filtros o periodos, hay que actualizar esta narrativa después de ejecutar de nuevo.
> Distingue siempre cuatro poblaciones: tablas completas, muestra de procesos, cohorte con etiqueta evaluable y empresas con positivos observados. Sus porcentajes no son intercambiables.

# 06 · EDA para recomendar oportunidades SECOP II a empresas

**Pregunta de producto:** dado el perfil de una empresa y una oportunidad recién publicada,
¿qué oportunidades merece la pena revisar para decidir si participar?

Este notebook continúa el ETL y consulta `secop.procesos`, `secop.contratos`,
`secop.proveedores`, `secop.grupos` y `gold.procesos_modelo` en PostgreSQL.
Todas las tablas de resultados, figuras e interpretaciones permanecen aquí. Las consultas son
de solo lectura. No se escriben CSV, imágenes, modelos, tablas SQL ni reportes externos.

**Qué puede observarse:** atributos del proceso, contratos firmados y catálogo actual de proveedores.
**Qué falta:** lista completa de participantes/no ganadores, decisiones de aplicar, clics,
requisitos habilitantes, documentos de la oferta y versiones históricas de sus cambios.
Un contrato adjudicado es evidencia positiva de interacción histórica; ausencia de contrato
no demuestra falta de encaje ni que la empresa perdió una licitación.

El análisis distingue: (1) calidad y cobertura, (2) relaciones y heterogeneidad,
(3) un modelo auxiliar de evidencia de contrato a 240 días y (4) un ranking experimental
empresa–oportunidad evaluado con interacciones positivas futuras. Ninguno estima por sí solo
la probabilidad de ganar de una empresa ni certifica elegibilidad legal.

### A01 · Preparación y reproducibilidad

**Por qué se hace.** Para que las cifras se puedan reconstruir y para saber a qué periodo y población pertenecen. Dos análisis con cortes o filtros distintos pueden llegar a conclusiones distintas aunque ambos estén bien calculados.

**Qué se hace.** Se establece la conexión, el corte del ETL, la semilla de muestreo, el horizonte de 240 días y las modalidades excluidas.

**Cómo se hace.** Las consultas se ejecutan en transacciones de solo lectura. Una semilla fija controla los componentes aleatorios; el hash del identificador selecciona siempre la misma muestra. Los códigos se tratan como texto y se extraen sus ocho dígitos finales para construir segmentos UNSPSC. Las funciones auxiliares centralizan consultas, normalización y los intervalos de Wilson.

In [ ]:
from pathlib import Path
import sys, re, unicodedata, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sqlalchemy import text

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
assert (ROOT / 'secop_pipeline').exists(), 'Ejecuta desde la raíz del proyecto o notebooks/.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from secop_pipeline.db import get_engine
from secop_pipeline.config import CUTOFF_DATE, START_DATE, TIPOS, SIN_COMPETENCIA
%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 4.5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .15,
                     'figure.dpi': 110, 'font.size': 10})
pd.set_option('display.max_columns', 20)
SEED = 20261007
rng = np.random.default_rng(SEED)
HORIZONTE = 240
engine = get_engine()
PARAMS = {'inicio': START_DATE.to_pydatetime(),
          'corte': (CUTOFF_DATE + pd.Timedelta(days=1)).to_pydatetime()}
# Listas procedentes de configuración local, escapadas como literales SQL.
EXCLUIDAS_SQL = ','.join("'" + x.lower().replace("'", "''") + "'" for x in SIN_COMPETENCIA)
SCOPE = f"lower(trim(modalidad_de_contratacion)) NOT IN ({EXCLUIDAS_SQL})"
def query(sql, params=None):
    with engine.connect() as conn:
        with conn.begin():
            conn.execute(text('SET TRANSACTION READ ONLY'))
            conn.execute(text("SET LOCAL statement_timeout = '240s'"))
            return pd.read_sql_query(text(sql), conn, params={**PARAMS, **(params or {})})

def say(s): display(Markdown(s))
def catcode(s):
    # V1.80121601 -> 80121601. Los códigos no son cantidades numéricas.
    return s.astype('string').str.extract(r'(\d{8})$', expand=False)
def norm(s):
    return s.astype('string').str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii').str.upper().str.strip().astype('string')
def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = np.divide(k, n, out=np.full_like(k, np.nan), where=n > 0)
    den = 1 + z*z/n
    mid = (p + z*z/(2*n))/den
    half = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return mid-half, mid+half
print('Corte de extracción:', CUTOFF_DATE.date(), '| semilla:', SEED)
print('El notebook usa pandas/NumPy/Matplotlib y SQL; no requiere scikit-learn o seaborn.')

**Conclusión del análisis A01**

La ejecución guardada usa el corte **21 de septiembre de 2026** y una ventana auxiliar de **240 días**. Las celdas terminaron con acceso a PostgreSQL y los resultados se conservaron dentro del notebook. Esta reproducibilidad corresponde a la fotografía actual: si la API o las tablas se actualizan, la misma consulta puede devolver otros datos. El horizonte de 240 días es una decisión experimental que debe compararse posteriormente con 180 y 365 días.

## 1. Unidad de análisis, objetivos e hipótesis

| Pregunta | Unidad | Señal observable / objetivo | Límite |
|---|---|---|---|
| ¿Qué oferta encaja con una empresa? | empresa × proceso × fecha | relevancia declarada, guardado o participación futura | aún no está disponible; es la etiqueta de producto que debemos recolectar |
| ¿El historial ayuda a recuperar oportunidades? | empresa × oportunidad | contrato futuro observado para esa empresa, dentro de una ventana | señal positiva sesgada hacia ganadores; no existen negativos fiables |
| ¿El proceso genera un contrato observado? | proceso/portafolio | contrato firmado entre publicación y +240 días | auxiliar; no representa encaje de una empresa |
| ¿Qué mide Gold? | proceso | `adjudicado_si OR tiene_contrato` | mezcla señales, usa estado final y no fija horizonte |

**Hipótesis predefinidas:**

- **H1:** afinidad de categoría histórica recupera más interacciones positivas futuras que popularidad global.
- **H2:** la cercanía geográfica del proveedor al comprador varía por segmento; no es universal.
- **H3:** la asociación presupuesto–contrato cambia al controlar modalidad; el agregado puede esconder mezcla de poblaciones.
- **H4:** los segmentos con concentración alta requieren una estrategia distinta para empresas nuevas; concentración no prueba barreras ni prácticas indebidas.
- **H5:** conocer a la entidad mejora recuperación, pero puede reducir descubrimiento. Se compara una ablación sin esta señal.
- **H6:** la disponibilidad de plazos y precio añade información a un modelo auxiliar sobre la prevalencia base, con evaluación futura.

Los gráficos exploratorios incluyen todo el corte disponible. Las variables y particiones del experimento
se declaran explícitamente; este EDA no constituye una validación externa independiente.
Toda elección posterior a observar resultados debe confirmarse con nuevas ventanas congeladas.

### A02 · Inventario, granularidad y modalidades

**Por qué se hace.** Antes de analizar frecuencias o entrenar un modelo hay que distinguir filas de entidades reales. Contar registros repetidos como procesos independientes altera distribuciones, tasas y estimaciones de incertidumbre.

**Qué se hace.** Se cuentan filas, identificadores distintos y claves nulas; también se identifica qué modalidades dominan la base y cuáles podrían aportar oportunidades a las que una empresa pueda aplicar.

**Cómo se hace.** SQL calcula `COUNT(*)`, `COUNT(DISTINCT clave)` y conteos condicionales. La diferencia entre filas e identificadores distintos indica filas adicionales por clave, pero no demuestra que sean copias idénticas. Para grupos la unidad de la tabla es la membresía, por lo que varios registros por grupo son esperables. Las modalidades se comparan como categorías nominales.

In [ ]:
inventario = []
keys = {'procesos':'id_del_proceso', 'contratos':'id_contrato',
        'proveedores':'codigo', 'grupos':None, 'procesos_modelo':'id_del_proceso'}
for schema, table in [('secop','procesos'),('secop','contratos'),('secop','proveedores'),
                      ('secop','grupos'),('gold','procesos_modelo')]:
    key = keys[table]
    unique = f'count(DISTINCT {key}) AS ids_distintos, count(*) FILTER (WHERE {key} IS NULL) AS ids_nulos' if key else 'NULL AS ids_distintos, NULL AS ids_nulos'
    rec = query(f'SELECT count(*) AS filas, {unique} FROM {schema}.{table}').iloc[0].to_dict()
    inventario.append({'tabla':f'{schema}.{table}', **rec})
inventario = pd.DataFrame(inventario)
display(inventario)
modalidades = query('SELECT modalidad_de_contratacion AS modalidad, count(*) AS n FROM secop.procesos GROUP BY 1 ORDER BY 2 DESC')
display(modalidades)
say('El universo para recomendar excluye las modalidades de la configuración comparando texto en minúsculas. '
    'Las modalidades «con ofertas» se conservan. Este filtro no certifica que todos los procesos sean accesibles '
    'a cualquier empresa; acuerdos marco, enajenaciones y precalificaciones necesitan reglas adicionales.')

**Conclusión del análisis A02**

Hay **9.070.082 filas de procesos y 8.646.130 identificadores diferentes**: 423.952 filas adicionales por identificador. Contratos tiene 295 filas adicionales por identificador; proveedores y Gold sí presentan una fila por clave en esta revisión. Se debe estudiar si los repetidos representan versiones, etapas o duplicados exactos antes de eliminarlos en el ETL.

La contratación directa y el régimen especial sin el calificativo «con ofertas» dominan el volumen. Por eso una recomendación de oportunidades necesita un universo explícito; usar todas las filas produciría un análisis dominado por modalidades que no necesariamente permiten concurrencia abierta. El filtro utilizado requiere después reglas de elegibilidad específicas.

### A03 · Cobertura del perfil, definición de la etiqueta y claves de cruce

**Por qué se hace.** El producto necesita conocer la actividad de la empresa y relacionarla con oportunidades. También necesita una etiqueta cuyo significado sea claro y enlaces que no asignen el mismo contrato a procesos incorrectos.

**Qué se hace.** Se revisan tres piezas: disponibilidad de la categoría del proveedor, acuerdo entre las señales que componen `objetivo` y cantidad de procesos que comparten un portafolio en Gold.

**Cómo se hace.** Se cuentan campos no nulos del catálogo. Se cruza `adjudicado_si` con `tiene_contrato` y se cuentan las cuatro combinaciones. Finalmente se agrupa por portafolio y se cuenta cuántos procesos distintos contiene. «Categoría informada» mide presencia, no validación semántica del código.

In [ ]:
catalogo = query("""SELECT count(*) AS filas, count(codigo_categoria_principal) AS categoria_informada,
 count(departamento) AS departamento_informado, count(fecha_creacion) AS fecha_informada
 FROM secop.proveedores""")
display(catalogo)
coverage = 100 * catalogo.categoria_informada.iloc[0] / catalogo.filas.iloc[0]
say(f'**Cobertura de categoría del catálogo: {coverage:.2f}%.** '
    'Un recomendador basado solamente en la categoría declarada dejaría sin perfil a muchos proveedores. '
    'El historial de contratos y un formulario de perfil son alternativas complementarias.')
signals = query('SELECT adjudicado_si,tiene_contrato,objetivo,count(*) AS n FROM gold.procesos_modelo GROUP BY 1,2,3 ORDER BY 1,2')
display(signals)
disagreement = signals.loc[signals.adjudicado_si != signals.tiene_contrato, 'n'].sum()/signals.n.sum()
say(f'Las dos señales de Gold discrepan en **{disagreement:.2%}** de las filas. '
    'No son etiquetas intercambiables. Adjudicar y firmar un contrato son eventos diferentes.')
portfolio_audit = query("""SELECT count(*) AS portafolios, count(*) FILTER(WHERE procesos>1) AS ambiguos,
 sum(procesos) FILTER(WHERE procesos>1) AS procesos_en_portafolios_ambiguos
 FROM (SELECT id_del_portafolio,count(DISTINCT id_del_proceso) AS procesos
       FROM gold.procesos_modelo WHERE id_del_portafolio IS NOT NULL GROUP BY 1) z""")
display(portfolio_audit)
say('Un enlace por portafolio puede asociar el mismo contrato a varios procesos. '
    'Más abajo se cuenta esta ambigüedad en todo el universo considerado y se excluye del experimento auxiliar y del backtest.')

**Conclusión del análisis A03**

Solo **12.097 de 1.619.281 proveedores (0,75%)** tienen categoría principal informada. Un perfil construido únicamente con ese campo tendría cobertura muy baja; se necesitan historial contractual y datos declarados por la empresa.

En Gold hay **22.778 procesos con contrato pero sin señal de adjudicación**, y **14.084 con señal de adjudicación pero sin contrato**: el desacuerdo total es **5,88%**. Puede reflejar tiempos, cobertura o diferencias entre eventos; no se interpreta automáticamente como error. El OR de ambas señales mide evidencia de resultado del proceso, no relevancia para una empresa.

Además, **3.646 portafolios de Gold agrupan varios procesos**, que suman 7.748 filas. Esto obliga a revisar la unidad de oportunidad y la relación entre etapas antes de usar el contrato como etiqueta individual.

## 2. Diccionario de variables y disponibilidad temporal

El tipo SQL no determina el uso estadístico: códigos UNSPSC, NIT e identificadores son nominales;
conteos son discretos; valores monetarios son continuos asimétricos; fechas definen el orden temporal.
No correlacionamos códigos de categorías convertidos a números.

La fotografía actual no garantiza qué precio, plazo, estado del proveedor o requisitos se conocían
en la publicación histórica. «Candidata al publicar» significa **hipótesis de disponibilidad pendiente de auditar**.

### A04 · Selección de variables según tipo y momento de disponibilidad

**Por qué se hace.** Un modelo puede parecer excelente si recibe datos que solo aparecen después del resultado. También puede producir asociaciones sin sentido si trata identificadores o categorías como cantidades.

**Qué se hace.** Se construye un diccionario de columnas con su tipo SQL y su papel analítico: predictor candidato, etiqueta, resultado posterior, señal que evoluciona durante el proceso o identificador de enlace.

**Cómo se hace.** Se leen los esquemas de PostgreSQL y se contrastan con reglas explícitas de disponibilidad. `objetivo` se separa de las entradas; adjudicación, ganador y contrato posterior se excluyen de los predictores. NIT y UNSPSC son nominales aunque contengan dígitos. Duraciones solo pueden combinarse tras homologar sus unidades. Un contrato pasado puede describir experiencia si su evento era anterior a la predicción.

In [ ]:
columns = query("""SELECT table_schema,table_name,column_name,data_type FROM information_schema.columns
 WHERE table_schema IN ('secop','gold') ORDER BY table_schema,table_name,ordinal_position""")
leak = set(['adjudicado','adjudicado_si','tiene_contrato','proceso_de_compra','estado_del_procedimiento',
 'estado_resumen','fecha_adjudicacion','nit_del_proveedor_adjudicado','codigoproveedor',
 'valor_total_adjudicacion','departamento_proveedor','ciudad_proveedor'])
dynamic = set(['fase','proveedores_invitados','proveedores_con_invitacion','visualizaciones_del',
 'proveedores_que_manifestaron','respuestas_al_procedimiento','respuestas_externas',
 'conteo_de_respuestas_a_ofertas','proveedores_unicos_con'])
def role(row):
    name, table = row.column_name, row.table_name
    if table in ('procesos','procesos_modelo'):
        if name=='objetivo': return 'Etiqueta heredada; no predictor'
        if name in leak: return 'Resultado posterior / fuente de etiqueta; excluir de X'
        if name in dynamic: return 'Evoluciona durante proceso; solo EDA retrospectivo'
        if name in ('id_del_proceso','id_del_portafolio'): return 'Identificación/cruce; excluir de X'
        return 'Candidata al publicar; auditar versión histórica'
    if table=='contratos': return 'Historial si firma < predicción; resultado si firma posterior'
    if table=='proveedores': return 'Catálogo actual; no asumir disponibilidad histórica'
    return 'Membresía actual de grupo; no asumir composición histórica'
columns['uso'] = columns.apply(role, axis=1)
display(columns.style.hide(axis='index'))
say('`duracion` requiere interpretar `unidad_de_duracion`; no se mezclan meses, días y años en una correlación. '
    'El modelo auxiliar usa una lista explícita de variables, sin identificadores ni resultados.')

**Conclusión del análisis A04**

Las entradas defendibles para un primer piloto son características de la oportunidad al publicarse y experiencia previa de la empresa. La fotografía disponible no prueba que los precios o plazos históricos sean sus versiones originales. Por eso se habla de **candidatas a predictor**, pendientes de auditar temporalmente. Las métricas de visitas y respuestas sirven para EDA retrospectivo, pero no entran en el modelo al publicar. La firma tampoco garantiza que el registro ya estuviera disponible públicamente en esa misma fecha.

## 3. Muestra reproducible y cruce sin multiplicar filas

Se toma **1/16 de los identificadores de proceso** mediante el primer carácter de MD5, desde el universo
de modalidades consideradas, incluyendo estados abiertos y terminados. Es una muestra determinista
por identificador, no un `LIMIT` por fecha. Se conserva una fila por identificador y se reportan los duplicados.

Los contratos se agregan antes del cruce. Se cuenta cuántos procesos diferentes comparten el portafolio
en todo este universo; no se atribuye una interacción específica cuando el cruce es ambiguo.
La elección entre filas duplicadas es determinista; no existe fecha de última modificación del proceso.
Se detectan discrepancias entre los atributos de entrada y se excluyen esos identificadores del modelo
auxiliar. Eso no demuestra que los demás atributos sean versiones históricas originales.
`contrato_240d` significa que existe una firma observada en [publicación, publicación +240 días].
Sin 240 días de seguimiento, o sin clave de cruce, la etiqueta queda desconocida.

### A05 · Muestreo, atribución del contrato y maduración de la etiqueta

**Por qué se hace.** Se necesita una muestra manejable que no favorezca los primeros registros de la tabla, una etiqueta con tiempo suficiente para observarse y un cruce que no multiplique oportunidades.

**Qué se hace.** Se selecciona una fracción reproducible de procesos; se revisan repetidos y atributos discordantes; se cuentan contratos por portafolio y se construye `contrato_240d`.

**Cómo se hace.** Se conserva el primer carácter de MD5 igual a `0`, aproximadamente 1/16 de los identificadores del universo. La elección entre filas repetidas es determinista. Los contratos se agregan por portafolio antes de enlazarlos. La etiqueta vale 1 cuando existe firma entre publicación y +240 días, y 0 cuando no se observa en esa ventana. Queda desconocida si falta seguimiento, hay enlace ambiguo, firmas anteriores a publicación o atributos de entrada discordantes. Se muestra la retención por modalidad para medir el sesgo creado por estas reglas.

In [ ]:
PROCESS_COLS = ['id_del_proceso','id_del_portafolio','nit_entidad','departamento_entidad',
 'ciudad_entidad','ordenentidad','modalidad_de_contratacion','tipo_de_contrato',
 'codigo_principal_de_categoria','precio_base','numero_de_lotes','duracion','unidad_de_duracion',
 'fecha_de_publicacion_del','fecha_de_recepcion_de','estado_del_procedimiento',
 'adjudicado','proveedores_unicos_con','respuestas_al_procedimiento','visualizaciones_del']
select_cols = ','.join(PROCESS_COLS)
# Una agregación sobre todo el universo evita evaluar cardinalidad solo dentro de la muestra.
portfolio_counts = query(f"""SELECT id_del_portafolio, count(DISTINCT id_del_proceso) AS n_procesos_portafolio
 FROM secop.procesos WHERE {SCOPE} AND id_del_portafolio IS NOT NULL GROUP BY 1""")
sample = query(f"""WITH sampled AS (
 SELECT {select_cols}, count(*) OVER(PARTITION BY id_del_proceso) AS repeticiones_id,
 row_number() OVER(PARTITION BY id_del_proceso ORDER BY fecha_de_publicacion_del,
 id_del_portafolio NULLS LAST, to_jsonb(p)::text) AS rn
 FROM secop.procesos p WHERE {SCOPE} AND id_del_proceso IS NOT NULL
 AND left(md5(id_del_proceso),1)='0' AND fecha_de_publicacion_del>=:inicio AND fecha_de_publicacion_del<:corte
 ) SELECT p.*, c.n_contratos,c.primera_firma,c.n_antes,c.n_240
 FROM sampled p LEFT JOIN LATERAL (
 SELECT count(*) AS n_contratos, min(fecha_de_firma) AS primera_firma,
 count(*) FILTER(WHERE fecha_de_firma < p.fecha_de_publicacion_del) AS n_antes,
 count(*) FILTER(WHERE fecha_de_firma>=p.fecha_de_publicacion_del
 AND fecha_de_firma<=p.fecha_de_publicacion_del + interval '240 days') AS n_240
 FROM secop.contratos c WHERE c.proceso_de_compra=p.id_del_portafolio
 ) c ON TRUE WHERE p.rn=1""")
sample = sample.merge(portfolio_counts, how='left', on='id_del_portafolio', validate='many_to_one')
variants=query(f"""SELECT id_del_proceso,count(DISTINCT ROW(id_del_portafolio,nit_entidad,
 departamento_entidad,modalidad_de_contratacion,tipo_de_contrato,codigo_principal_de_categoria,
 precio_base,numero_de_lotes,fecha_de_publicacion_del,fecha_de_recepcion_de)) AS variantes_predictores
 FROM secop.procesos WHERE {SCOPE} AND left(md5(id_del_proceso),1)='0'
 AND fecha_de_publicacion_del>=:inicio AND fecha_de_publicacion_del<:corte GROUP BY 1""")
sample=sample.merge(variants,on='id_del_proceso',how='left',validate='one_to_one')
for name in ['fecha_de_publicacion_del','fecha_de_recepcion_de','primera_firma']:
    sample[name] = pd.to_datetime(sample[name], errors='coerce')
for name in ['precio_base','numero_de_lotes','proveedores_unicos_con','respuestas_al_procedimiento','visualizaciones_del']:
    sample[name] = pd.to_numeric(sample[name],errors='coerce').replace([np.inf,-np.inf],np.nan)
sample['categoria8'] = catcode(sample.codigo_principal_de_categoria)
sample['segmento'] = sample.categoria8.str[:2]
sample['familia'] = sample.categoria8.str[:4]
sample['plazo_bruto'] = (sample.fecha_de_recepcion_de-sample.fecha_de_publicacion_del).dt.total_seconds()/86400
sample['log10_precio'] = np.log10(sample.precio_base.where(sample.precio_base>0))
sample['anio'] = sample.fecha_de_publicacion_del.dt.year
sample['maduro'] = sample.fecha_de_publicacion_del <= CUTOFF_DATE-pd.Timedelta(days=HORIZONTE)
sample['enlace_evaluable'] = sample.id_del_portafolio.notna() & sample.n_procesos_portafolio.eq(1)
eligible = sample.maduro & sample.enlace_evaluable & sample.n_antes.eq(0) & sample.variantes_predictores.eq(1)
sample['contrato_240d'] = np.where(eligible, sample.n_240.gt(0).astype(int), np.nan)
display(pd.Series({'muestra_procesos':len(sample),'filas_repetidas_extra':int((sample.repeticiones_id-1).sum()),
 'portafolio_no_informado':int(sample.id_del_portafolio.isna().sum()),
 'portafolio_ambiguo':int(sample.n_procesos_portafolio.gt(1).sum()),
 'id_con_predictores_discordantes':int(sample.variantes_predictores.gt(1).sum()),
 'con_firmas_anteriores_a_publicacion':int(sample.n_antes.gt(0).sum()),
 'sin_horizonte_completo':int((~sample.maduro).sum()),
 'etiquetas_evaluables':int(sample.contrato_240d.notna().sum())},name='n'))
retention=sample.groupby('modalidad_de_contratacion').agg(muestra=('id_del_proceso','size'),
    evaluables=('contrato_240d','count'),ambiguos=('n_procesos_portafolio',lambda s:s.gt(1).sum()))
retention['retencion_pct']=100*retention.evaluables/retention.muestra
display(retention.sort_values('muestra',ascending=False))
say('La ausencia de un contrato vinculado es **ausencia de evidencia**, no cancelación demostrada. '
    'Los contratos sin fecha de firma o excluidos por el corte pueden causar falsos negativos. '
    'Se excluyen casos con firmas anteriores a publicación porque cuestionan la semántica del enlace.')
say('**Sesgo de cobertura:** un portafolio puede contener fases, versiones o reinicios legítimos. '
    'Excluirlos evita atribuciones falsas, pero puede eliminar modalidades enteras. El modelo auxiliar '
    'solo describe las modalidades con soporte en la tabla anterior. Para ampliar cobertura hay que '
    'reconstruir el ciclo del portafolio y la clave correcta de la oportunidad, no duplicar la etiqueta entre fases.')

**Conclusión del análisis A05**

La muestra contiene **54.709 procesos**; **23.268** comparten un portafolio con otros procesos del universo, y **5.365** no tienen todavía 240 días de seguimiento. Las condiciones se superponen, por lo que no deben sumarse como exclusiones independientes. Quedan **28.376 etiquetas evaluables**. No se encontraron discrepancias en el conjunto de predictores comparados entre filas del mismo identificador; eso no valida todas sus columnas ni su historia.

El efecto de selección es fuerte: se retiene **88,9% de mínima cuantía**, pero solo **3,0% de licitación pública** y **1,0% de selección abreviada de menor cuantía**. El modelo describe principalmente mínima cuantía y modalidades directa/especial con ofertas. Para ampliar cobertura hay que reconstruir el ciclo del portafolio; no sería correcto generalizar estas tasas a todas las licitaciones ni llamar «fracaso» a la ausencia de un contrato enlazado.

### A06 · Faltantes, valores imposibles y estabilidad temporal

**Por qué se hace.** Un campo completo puede contener valores inválidos. Además, los patrones de registro pueden cambiar por año y confundirse con cambios económicos o de comportamiento.

**Qué se hace.** Se calcula cobertura, cardinalidad, precios no positivos y plazos incoherentes; se visualizan el volumen y la falta de datos por año.

**Cómo se hace.** Cada porcentaje usa como denominador la muestra correspondiente. Se calcula el plazo bruto restando las fechas, sin recortarlo previamente. Las líneas por año permiten identificar cambios de captura. Se separan las reglas de calidad de las transformaciones utilizadas para modelar o visualizar.

In [ ]:
quality_cols = ['precio_base','numero_de_lotes','fecha_de_recepcion_de','segmento',
                'departamento_entidad','nit_entidad','proveedores_unicos_con']
quality = pd.DataFrame({'nulos_pct':sample[quality_cols].isna().mean()*100,
                        'distintos':sample[quality_cols].nunique()})
display(quality)
display(pd.Series({'precio_cero':int(sample.precio_base.eq(0).sum()),
 'precio_negativo':int(sample.precio_base.lt(0).sum()),
 'plazo_negativo':int(sample.plazo_bruto.lt(0).sum()),
 'plazo_mayor_365':int(sample.plazo_bruto.gt(365).sum())}))
coverage_time = sample.groupby('anio').agg(n=('id_del_proceso','size'),
                    precio_faltante=('precio_base',lambda x:x.isna().mean()),
                    plazo_faltante=('plazo_bruto',lambda x:x.isna().mean()),
                    segmento_faltante=('segmento',lambda x:x.isna().mean()))
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(coverage_time.index,coverage_time.n,marker='o'); ax[0].set(title='Cobertura temporal de la muestra',xlabel='Año de publicación',ylabel='Procesos muestreados')
(coverage_time.iloc[:,1:]*100).plot(ax=ax[1],marker='o'); ax[1].set(title='Faltantes por año: posible cambio de captura',xlabel='Año',ylabel='% de procesos')
plt.tight_layout(); plt.show()
say('2026 es un año parcial. Cambios de volumen o faltantes pueden reflejar adopción de SECOP y cambios '
    'de captura. Los precios son COP nominales; no se atribuye su crecimiento a aumentos reales sin deflactar.')

**Conclusión del análisis A06**

Aunque `precio_base` no tiene nulos en la muestra, hay **1.004 ceros y 3 negativos**. También hay **55 plazos negativos y 3 superiores a 365 días**. Segmento tiene **4,47%** de faltantes y departamento **3,06%**. Por tanto, «no nulo» no equivale a dato utilizable.

Para gráficos logarítmicos se omiten precios no positivos; para modelar se tratan como ausencias con un indicador. Los plazos extraños deben revisarse antes de interpretarlos. **2026 está incompleto**, y los precios son nominales: el gráfico temporal por sí solo no prueba caída del mercado ni crecimiento real del presupuesto.

## 4. Distribuciones y relaciones que respetan el tipo de variable

Para dinero se usa escala logarítmica y cuantiles; para categorías, frecuencias y tasas;
para dos medidas numéricas, densidad y dispersión. Una nube de 70.000 puntos superpuestos
oculta patrones: `hexbin` muestra densidad y el scatter usa una submuestra declarada.
No se rellenan precios faltantes con cero para estas figuras.

### A07 · Distribuciones de dinero, lotes, plazos y modalidades

**Por qué se hace.** La distribución determina qué resumen y escala tienen sentido. Una media dominada por contratos grandes no representa la oportunidad típica.

**Qué se hace.** Se obtienen cuantiles, un histograma de presupuesto positivo, frecuencias de modalidades y diagramas de caja por modalidad.

**Cómo se hace.** La tabla de cuantiles usa los procesos con etiqueta evaluable. El histograma y las cajas usan los precios positivos de la muestra más amplia; sus denominadores son distintos. `log10(COP)` permite ver varios órdenes de magnitud: 7 corresponde a 10 millones y 8 a 100 millones. Las cajas muestran mediana y rango intercuartílico; omitir los puntos atípicos del dibujo no los elimina de la base. Las categorías se muestran mediante barras, no como números continuos.

In [ ]:
observed = sample.loc[sample.contrato_240d.notna()].copy()
display(observed[['precio_base','numero_de_lotes','plazo_bruto']].describe(percentiles=[.01,.25,.5,.75,.95,.99]).T)
fig,ax=plt.subplots(1,2,figsize=(12,4))
x=sample.log10_precio.dropna()
ax[0].hist(x,bins=55,color='#246B8E'); ax[0].set(title=f'Presupuesto positivo · n={len(x):,}',xlabel='log10(precio base en COP)',ylabel='Procesos')
counts=sample.modalidad_de_contratacion.value_counts().head(10).sort_values()
ax[1].barh(counts.index,counts.values,color='#3A917B'); ax[1].set(title='Modalidades de la muestra',xlabel='Procesos')
plt.tight_layout(); plt.show()
topmods=sample.modalidad_de_contratacion.value_counts().head(5).index.tolist()
fig,ax=plt.subplots(figsize=(11,4))
ax.boxplot([sample.loc[sample.modalidad_de_contratacion.eq(m),'log10_precio'].dropna() for m in topmods],
           vert=False,showfliers=False)
ax.set_yticks(range(1,len(topmods)+1),topmods)
ax.set(title='Escala del presupuesto por modalidad',xlabel='log10(COP); cajas P25–P75, línea mediana; atípicos no dibujados')
plt.tight_layout(); plt.show()

**Conclusión del análisis A07**

En la cohorte evaluable, la mediana del presupuesto es aproximadamente **28 millones COP**, mientras que la media ronda **432 millones COP**. La gran diferencia indica una cola de valores elevados; se recomienda describir escala con mediana, cuantiles y logaritmos.

Las cajas muestran presupuestos menores en mínima cuantía y distribuciones diferentes en las otras modalidades. El ajuste entre empresa y oferta debe considerar modalidad y escala, no un presupuesto promedio global. El percentil 95 de `numero_de_lotes` es cero: ese campo aporta poca variación y se debe aclarar qué significa cero antes de interpretarlo como ausencia efectiva de lotes.

### A08 · Scatterplot y densidad: presupuesto frente a plazo

**Por qué se hace.** Se busca comprobar si presupuesto y tiempo para preparar la oferta forman regiones útiles para segmentar oportunidades, y si esas dos variables separan visualmente los resultados.

**Qué se hace.** Se dibuja la densidad conjunta de precio y plazo y una nube coloreada por evidencia de contrato a 240 días.

**Cómo se hace.** El hexbin agrupa observaciones próximas en celdas; su color representa conteo en escala logarítmica y reduce la superposición. El scatter toma hasta 3.500 casos con etiqueta de una submuestra reproducible y usa transparencia. Se muestran precios positivos y plazos de 0 a 365 días; cada panel deja que su eje vertical se ajuste al rango que contiene, por lo que hay que leer sus escalas.

In [ ]:
plot_data=sample.loc[sample.plazo_bruto.between(0,365)&sample.log10_precio.notna()].copy()
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
hb=ax[0].hexbin(plot_data.log10_precio,plot_data.plazo_bruto,gridsize=38,bins='log',mincnt=1,cmap='viridis')
fig.colorbar(hb,ax=ax[0],label='Conteo por celda (escala log)')
ax[0].set(title=f'Presupuesto y plazo · n={len(plot_data):,}',xlabel='log10(COP)',ylabel='Días entre publicación y recepción')
scatter=plot_data.loc[plot_data.contrato_240d.notna()].sample(n=min(3500,int(plot_data.contrato_240d.notna().sum())),random_state=SEED)
for target,color in [(0,'#B86F52'),(1,'#246B8E')]:
    z=scatter[scatter.contrato_240d==target]
    ax[1].scatter(z.log10_precio,z.plazo_bruto,s=9,alpha=.25,c=color,label=f'Contrato 240d={target}')
ax[1].set(title='Scatter: superposición de resultados',xlabel='log10(COP)',ylabel='Plazo previsto (días)'); ax[1].legend()
plt.tight_layout(); plt.show()
say(f'El gráfico cubre {len(plot_data)/len(sample):.1%} de la muestra: se omiten faltantes, '
    'precios no positivos y plazos fuera de 0–365 solo para visualizar. El modelo conserva indicadores '
    'de invalidez. La separación visual no implica causalidad ni anticipa el éxito de una empresa.')

**Conclusión del análisis A08**

El filtro gráfico conserva **98,1% de la muestra**. Se observa una concentración de procesos en plazos cortos y presupuestos intermedios, junto con una cola de valores y plazos altos. En el scatter los dos resultados se superponen ampliamente: precio y plazo por sí solos no ofrecen una frontera visual clara para clasificar contratos.

La figura justifica explorar relaciones no lineales y diferencias por modalidad. No justifica una regla del tipo «presupuesto alto implica adjudicación» ni permite medir el encaje de una empresa, porque todavía no incorpora su perfil.

### A09 · Tasas por modalidad e incertidumbre

**Por qué se hace.** Una tasa general puede esconder que los mecanismos y tiempos de contratación son diferentes. También hay que mostrar cuánto soporte tiene cada proporción.

**Qué se hace.** Se estima la fracción de procesos con contrato observado en 240 días para cada modalidad y se añade un intervalo descriptivo.

**Cómo se hace.** La tasa es `k/n`, con `k` positivos y `n` etiquetas observables. Se requieren al menos 100 casos por modalidad. El intervalo Wilson de 95% expresa incertidumbre binomial bajo independencia; no corrige por entidades repetidas, sesgo de selección ni errores de cruce.

In [ ]:
def rate_table(frame, group, target='contrato_240d', min_n=100):
    z=frame.groupby(group,observed=True)[target].agg(n='count',k='sum').query('n>=@min_n').copy()
    z['tasa']=z.k/z.n; z['lo'],z['hi']=wilson(z.k,z.n)
    return z.sort_values('n',ascending=False)
rates=rate_table(observed,'modalidad_de_contratacion')
display(rates)
z=rates.sort_values('tasa')
fig,ax=plt.subplots(figsize=(10,5))
ax.errorbar(z.tasa,z.index,xerr=[z.tasa-z.lo,z.hi-z.tasa],fmt='o',color='#246B8E',capsize=3)
ax.set(title='Evidencia de contrato en 240 días por modalidad',xlabel='Proporción e IC Wilson 95% (descriptivo)',xlim=(0,1))
plt.tight_layout(); plt.show()
say('Los IC Wilson suponen observaciones independientes; procesos de una misma entidad pueden estar '
    'correlacionados. Se usan para lectura descriptiva, no para afirmar significancia causal.')

**Conclusión del análisis A09**

Las tasas observadas son aproximadamente **80,2% en mínima cuantía**, **84,9% en directa con ofertas** y **71,5% en régimen especial con ofertas**. Son tasas de la cohorte conservada, no probabilidades de ganar de una empresa.

Los **121 casos de selección abreviada de menor cuantía tienen 0 contratos enlazados**. Esa modalidad retuvo menos del 1% de su muestra, por lo que este cero exige investigar cobertura y claves; sería incorrecto concluir que no produce contratos o descartarla del producto. Un intervalo estrecho no resuelve un sesgo sistemático de enlace.

## 5. Asociaciones y patrones ocultos: mezcla de modalidades, plazos y segmentos

Spearman mide asociación monótona entre numéricas; la correlación punto-biserial relaciona una
numérica con una etiqueta binaria; V de Cramér mide asociación entre nominales. Las medidas
responden preguntas distintas y no son intercambiables. No se usa Pearson sobre UNSPSC o NIT.
Las variables de interacción posteriores a publicación solo se exploran retrospectivamente.

### A10 · Correlaciones, asociaciones categóricas y mezcla de modalidades

**Por qué se hace.** Se quiere saber qué relaciones merecen explorarse y si una asociación agregada se mantiene dentro de poblaciones comparables.

**Qué se hace.** Se estudian asociaciones entre numéricas, entre categorías y la etiqueta, y entre cada numérica y el resultado binario. Después se compara la relación precio–resultado por modalidad.

**Cómo se hace.** Spearman correlaciona los rangos de las numéricas y detecta relaciones monótonas; se muestran los tamaños por pareja. La punto-biserial es Pearson entre una numérica y la etiqueta 0/1. V de Cramér corregida utiliza una tabla de contingencia entre categorías y resultado; no tiene signo. Se agrupan categorías con soporte escaso. Las variables de respuestas/proveedores posteriores a publicación entran solo en la exploración retrospectiva. Los valores de estas medidas no se comparan como si fueran una misma escala causal.

In [ ]:
numeric=['log10_precio','numero_de_lotes','plazo_bruto','proveedores_unicos_con','respuestas_al_procedimiento']
# rank + Pearson equivale a Spearman y evita dependencias adicionales.
corr=observed[numeric].rank().corr(min_periods=150)
pairs=observed[numeric].notna().astype(int).T @ observed[numeric].notna().astype(int)
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(corr,vmin=-1,vmax=1,cmap='RdBu_r')
ax.set_xticks(range(len(numeric)),numeric,rotation=60,ha='right'); ax.set_yticks(range(len(numeric)),numeric)
for i in range(len(numeric)):
    for j in range(len(numeric)):
        ax.text(j,i,f'{corr.iloc[i,j]:.2f}',ha='center',va='center',fontsize=9)
fig.colorbar(im,ax=ax,label='Spearman'); ax.set_title('Asociaciones numéricas; señales posteriores incluidas solo para EDA')
plt.tight_layout(); plt.show(); display(pairs)

def cramers_v(a,b):
    ct=pd.crosstab(a,b).to_numpy(float); n=ct.sum()
    if n<=1 or min(ct.shape)<2: return np.nan
    expected=ct.sum(1)[:,None]*ct.sum(0)[None,:]/n
    chi=((ct-expected)**2/expected).sum(); r,k=ct.shape
    phi=max(0,chi/n-(k-1)*(r-1)/(n-1))
    den=min(k-1-(k-1)**2/(n-1),r-1-(r-1)**2/(n-1))
    return np.sqrt(phi/den) if den>0 else np.nan
associations=[]
for col in ['modalidad_de_contratacion','tipo_de_contrato','segmento','departamento_entidad']:
    a=observed[col].fillna('SIN_DATO'); freq=a.value_counts()
    a=a.where(a.isin(freq[freq>=100].index),'OTRAS')
    associations.append({'variable':col,'medida':'V Cramér corregida','valor':cramers_v(a,observed.contrato_240d),'n':len(a)})
for col in ['log10_precio','numero_de_lotes','plazo_bruto']:
    z=observed[[col,'contrato_240d']].dropna()
    associations.append({'variable':col,'medida':'Punto-biserial (Pearson)','valor':z.corr().iloc[0,1],'n':len(z)})
display(pd.DataFrame(associations))

mix=[]
for group,z in [('TOTAL',observed)]+list(observed.groupby('modalidad_de_contratacion')):
    z=z[['log10_precio','contrato_240d']].dropna()
    if len(z)>=150: mix.append({'modalidad':group,'n':len(z),'r_precio_objetivo':z.corr().iloc[0,1]})
mix=pd.DataFrame(mix); display(mix)
overall=mix.loc[mix.modalidad.eq('TOTAL'),'r_precio_objetivo'].iloc[0]
reversals=mix[(mix.modalidad!='TOTAL') & (mix.r_precio_objetivo*overall<0)]
say(f'**H3:** asociación agregada precio–contrato = {overall:.3f}; '
    f'{len(reversals)} modalidades tienen signo opuesto con soporte mínimo de 150. '
    'Comparar también magnitudes: una atenuación agregada puede esconder composición de modalidades. '
    'No se afirma paradoja de Simpson ni efecto causal del presupuesto.')

**Conclusión del análisis A10**

Entre las asociaciones nominales examinadas, modalidad presenta la mayor V de Cramér (**0,233**), por encima de tipo de contrato (**0,072**), segmento (**0,064**) y departamento (**0,053**). Entre las numéricas, el precio tiene una asociación agregada pequeña con la etiqueta (**0,026**), lotes está casi en cero (**0,006**) y plazo es negativo (**−0,096**).

Dentro de las modalidades con suficiente soporte, la relación precio–resultado aumenta a **0,069–0,109** y mantiene signo positivo. La agregación atenúa la relación; **no se observó inversión de signo**, por lo que no se afirma una paradoja de Simpson. Se justifica explorar interacciones por modalidad, no convertir estas asociaciones en reglas causales. La falta de asociación marginal tampoco descarta un efecto no lineal o condicionado a otras variables.

### A11 · Interacción entre escala del presupuesto y plazo

**Por qué se hace.** Dos variables pueden aportar información conjuntamente aunque su relación individual con el resultado sea débil.

**Qué se hace.** Se compara la tasa de contrato entre combinaciones de quintil de precio y tramo de plazo.

**Cómo se hace.** Los precios positivos se dividen en cinco grupos de igual frecuencia aproximada, usando logaritmos. Los plazos se agrupan en intervalos. Cada celda muestra tasa y número de observaciones; se ocultan grupos con menos de 80 casos. Los quintiles pertenecen a esta muestra y no equivalen a umbrales comerciales universales.

In [ ]:
z=observed.loc[observed.log10_precio.notna()&observed.plazo_bruto.between(0,365)].copy()
z['quintil_precio']=pd.qcut(z.log10_precio,5,duplicates='drop')
z['plazo_grupo']=pd.cut(z.plazo_bruto,[-.1,3,7,15,30,365],labels=['0–3','>3–7','>7–15','>15–30','>30–365'])
cross=z.groupby(['quintil_precio','plazo_grupo'],observed=True).contrato_240d.agg(['mean','count'])
matrix=cross['mean'].unstack().where(cross['count'].unstack()>=80)
fig,ax=plt.subplots(figsize=(9,5)); im=ax.imshow(matrix,vmin=0,vmax=1,cmap='YlGnBu',aspect='auto')
ax.set_xticks(range(len(matrix.columns)),matrix.columns); ax.set_yticks(range(len(matrix.index)),[f'Q{i+1}' for i in range(len(matrix.index))])
for i in range(len(matrix.index)):
    for j in range(len(matrix.columns)):
        val=matrix.iloc[i,j]; n=cross['count'].unstack().reindex_like(matrix).iloc[i,j]
        ax.text(j,i,f'{val:.0%} / n={n:.0f}' if pd.notna(val) else 'n<80',ha='center',va='center',fontsize=8)
ax.set(title='Interacción exploratoria: presupuesto × plazo previsto',xlabel='Plazo (días)',ylabel='Quintil de log10(precio)')
fig.colorbar(im,ax=ax,label='Proporción con contrato observado ≤240 días'); plt.tight_layout(); plt.show()
say('Los quintiles son globales a esta muestra; el mapa mezcla modalidades. Celdas con menos de 80 '
    'observaciones se ocultan. Una región de tasa alta puede corresponder a un tipo de contratación específico.')

**Conclusión del análisis A11**

En la figura guardada, el quintil superior presenta tasas cercanas a **85% en 0–3 días**, pero de aproximadamente **31% en más de 15–30 días** y **23% en más de 30–365 días**. Estas últimas celdas tienen solo 335 y 125 observaciones.

Es un patrón para investigar, no evidencia de que dar más tiempo reduzca la contratación. Pueden cambiar la modalidad, el tipo de proceso, la calidad del enlace o la ventana necesaria para firmar. El siguiente contraste debe repetirse dentro de modalidades y con otro horizonte; muchas combinaciones no tienen soporte suficiente para extraer una regla.

## 6. Proveedores, territorio, concentración y grupos

Se investigan posibles diferencias de acceso. La concentración se calcula sobre **conteos de contratos**
por proveedor, dentro de segmento UNSPSC y para firmas 2024–2025 en las modalidades consideradas.
No es participación por valor ni un indicador de conducta irregular. Se excluyen claves/categorías ausentes.
El HHI es la suma de cuotas al cuadrado (0–1); su inverso es el número efectivo de proveedores.

### A12 · Concentración por segmento UNSPSC

**Por qué se hace.** Un segmento puede mostrar muchas empresas registradas, pero concentrar sus contratos en unas pocas. Esa diferencia importa al explorar mercados y oportunidades nuevas.

**Qué se hace.** Se mide concentración entre códigos de proveedor, tamaño del segmento y participación del proveedor de mayor volumen.

**Cómo se hace.** Se cuentan contratos distintos firmados en 2024–2025 dentro del universo de modalidades, agrupados por segmento y proveedor. Si `s_i` es la fracción de contratos de un proveedor, `HHI = suma(s_i²)`. `1/HHI` expresa cuántos proveedores de igual tamaño producirían esa concentración. El scatter usa ejes logarítmicos por la amplitud de escalas; color y tamaño reflejan la cuota del líder. Se exige al menos 100 contratos por segmento.

In [ ]:
concentration=query(f"""WITH base AS (
 SELECT id_contrato,codigo_proveedor,left(substring(codigo_de_categoria_principal from '([0-9]{{8}})$'),2) AS segmento
 FROM secop.contratos WHERE fecha_de_firma>='2024-01-01' AND fecha_de_firma<'2026-01-01'
 AND {SCOPE} AND codigo_proveedor IS NOT NULL), counts AS (
 SELECT segmento,codigo_proveedor,count(DISTINCT id_contrato)::float AS n FROM base
 WHERE segmento IS NOT NULL GROUP BY 1,2)
 SELECT segmento,sum(n) AS contratos,count(*) AS proveedores,
 sum(n*n)/power(sum(n),2) AS hhi,max(n)/sum(n) AS cuota_lider
 FROM counts GROUP BY 1 HAVING sum(n)>=100 ORDER BY hhi DESC""")
concentration['proveedores_efectivos']=1/concentration.hhi
display(concentration)
fig,ax=plt.subplots(figsize=(9,5))
points=ax.scatter(concentration.contratos,concentration.hhi,s=35+120*concentration.cuota_lider,
                  c=concentration.cuota_lider,cmap='plasma',alpha=.8)
ax.set_xscale('log'); ax.set_yscale('log')
for r in concentration.head(6).itertuples(): ax.annotate(r.segmento,(r.contratos,r.hhi),xytext=(4,4),textcoords='offset points')
ax.set(title='H4 · Concentración y tamaño del segmento',xlabel='Contratos 2024–2025 (log)',ylabel='HHI por conteos (log)')
fig.colorbar(points,ax=ax,label='Cuota del mayor proveedor'); plt.tight_layout(); plt.show()
say('Las diferencias de tamaño y especialización importan: comparar HHI de un segmento pequeño con uno '
    'muy amplio sin contexto puede inducir conclusiones erróneas. Analizar el segmento a 4/6 dígitos es una sensibilidad futura.')
lead=concentration.iloc[0]
say(f'El segmento {lead.segmento} tiene {int(lead.proveedores):,} códigos de proveedor, pero '
    f'solo {lead.proveedores_efectivos:.1f} proveedores efectivos por HHI. Su mayor proveedor concentra '
    f'{lead.cuota_lider:.1%} de los contratos del segmento. Es una señal para estudiar especialización; '
    'consolidar códigos por identidad empresarial antes de interpretar concentración de empresas.')

**Conclusión del análisis A12**

El segmento **84** reúne 4.450 contratos y 830 códigos de proveedor, pero equivale a solo **8,6 proveedores de igual participación**; el mayor reúne **24,4%** de sus contratos. Tener muchos proveedores nominales no implica una distribución equilibrada.

Esto orienta una investigación sobre especialización y estructura del mercado. No demuestra barreras de entrada, favoritismo ni baja probabilidad de éxito. El indicador utiliza conteos, no valor; además, diferentes códigos pueden pertenecer a una misma identidad empresarial. Se deben homologar empresas y contrastar familias UNSPSC antes de usar concentración como señal del producto.

### A13 · Territorio del comprador y domicilio del proveedor

**Por qué se hace.** Un filtro geográfico rígido podría eliminar oportunidades adecuadas si ciertos sectores operan habitualmente fuera de su domicilio.

**Qué se hace.** Se compara la frecuencia de contratos con coincidencia de departamento entre el registro contractual y el proveedor, por segmento.

**Cómo se hace.** Se toma una muestra por hash de contratos 2024–2025, se cruza por código de proveedor y se normalizan acentos, espacios y mayúsculas. Se calcula la proporción solo entre registros con ambos departamentos informados. Se muestran hasta 15 segmentos con mayor soporte entre los que tienen al menos 100 comparables, junto con intervalos Wilson.

In [ ]:
geo=query(f"""WITH cs AS (
 SELECT codigo_proveedor,departamento,left(substring(codigo_de_categoria_principal from '([0-9]{{8}})$'),2) AS segmento
 FROM secop.contratos WHERE {SCOPE} AND fecha_de_firma>='2024-01-01' AND fecha_de_firma<'2026-01-01'
 AND left(md5(coalesce(id_contrato,'')),1)='0')
 SELECT c.*,p.departamento AS departamento_proveedor,p.tipo_empresa,p.espyme
 FROM cs c LEFT JOIN secop.proveedores p ON p.codigo=c.codigo_proveedor""")
geo['d_entidad']=norm(geo.departamento); geo['d_proveedor']=norm(geo.departamento_proveedor)
geo['comparable']=geo.d_entidad.notna()&geo.d_proveedor.notna()
geo['local']=np.where(geo.comparable,geo.d_entidad.eq(geo.d_proveedor).fillna(False).astype(int),np.nan)
georates=rate_table(geo,'segmento',target='local',min_n=100).head(15)
display(georates)
fig,ax=plt.subplots(figsize=(10,4))
ax.errorbar(georates.index,georates.tasa,yerr=[georates.tasa-georates.lo,georates.hi-georates.tasa],fmt='o',capsize=3)
ax.set(title=f'H2 · Mismo departamento por segmento · cobertura {geo.comparable.mean():.1%}',
       xlabel='Segmento UNSPSC (categoría nominal)',ylabel='Proporción entre registros comparables',ylim=(0,1))
plt.tight_layout(); plt.show()
say('Se compara `contratos.departamento` con el domicilio **actual** del proveedor. Verificar la semántica '
    'del primero y homologar nombres territoriales antes de usarlo operacionalmente; la igualdad textual '
    'no mide distancia ni capacidad de operar fuera del domicilio. No se incorpora este catálogo actual al backtest histórico.')

**Conclusión del análisis A13**

Hay heterogeneidad: en el segmento **84** coincide el departamento en **26,5%** de los 211 casos comparables, mientras que en el **90** coincide en **86,2%** de 276 casos. Esto cuestiona que «mismo departamento» deba ser un requisito universal del recomendador.

La igualdad textual no representa distancia, lugar de ejecución ni capacidad operativa. El domicilio del catálogo es actual y puede haber cambiado; también deben homologarse alias territoriales y confirmar qué representa el departamento del contrato. La señal se utiliza para explorar preferencias geográficas configurables, no como predictor histórico validado ni como filtro excluyente.

### A14 · Grupos, composición y participación

**Por qué se hace.** Las empresas pueden participar mediante grupos. Atribuir todo el historial del grupo a cada miembro inflaría experiencia y mezclaría composiciones de fechas distintas.

**Qué se hace.** Se analiza cuántos miembros tiene cada grupo, repetición de participantes, líderes y suma de porcentajes de participación.

**Cómo se hace.** Se agrega por código de grupo y se cuentan participantes distintos. Se suman participaciones solo para revisar consistencia; la comprobación 99–101 se interpreta sobre grupos con todos esos valores informados. La barra de tamaño usa números enteros y agrupa diez o más miembros; el eje vertical logarítmico permite ver tamaños poco frecuentes.

In [ ]:
groups=query("""SELECT codigo_grupo,count(*) AS filas,count(DISTINCT codigo_participante) AS miembros,
 sum(participacion) AS suma_participacion,count(participacion) AS participaciones_informadas,
 count(*) FILTER(WHERE lower(trim(es_lider_del_grupo)) IN ('si','sí','true','1')) AS lideres
 FROM secop.grupos WHERE codigo_grupo IS NOT NULL GROUP BY 1""")
display(pd.Series({'grupos':len(groups),'con_miembros_repetidos':int(groups.filas.gt(groups.miembros).sum()),
 'con_mas_un_lider':int(groups.lideres.gt(1).sum()),
 'participacion_completa':int(groups.participaciones_informadas.eq(groups.filas).sum()),
 'suma_fuera_99_101_con_datos_completos':int((groups.participaciones_informadas.eq(groups.filas)&~groups.suma_participacion.between(99,101)).sum())}))
fig,ax=plt.subplots(figsize=(9,4)); gcounts=groups.miembros.clip(upper=10).value_counts().sort_index()
ax.bar(gcounts.index,gcounts.values,color='#3A917B'); ax.set_yscale('log')
ax.set(title='Tamaño de los grupos: distribución discreta',xlabel='Miembros distintos (10 agrupa 10 o más)',ylabel='Grupos (escala log)')
plt.tight_layout(); plt.show()
say('No se suman automáticamente contratos del grupo a cada integrante: duplicaría experiencia y puede '
    'usar membresía futura. Una extensión exige composición y participaciones vigentes en la fecha de la oferta, '
    'y reglas sobre experiencia atribuible. Las anomalías son casos de revisión, no errores confirmados.')

**Conclusión del análisis A14**

Se identifican **321.303 grupos**, de los cuales **9.135** presentan más filas que participantes distintos. Hay **939 grupos con participación completa cuya suma queda fuera de 99–101**. No se detectaron grupos con más de un líder bajo los valores de texto reconocidos por la regla, lo cual no demuestra que todos tengan líder válido.

No se puede asumir que estas anomalías sean errores: pueden existir versiones o registros repetidos que deben revisarse. La extensión del perfil empresarial mediante grupos requiere membresías fechadas, reglas de atribución de experiencia y deduplicación; se pospone su uso como predictor hasta disponer de esa trazabilidad.

## 7. Modelo auxiliar interpretable: contrato observado a 240 días

Sirve para explorar H3/H6, no para seleccionar empresas ganadoras. Se excluyen portafolios ambiguos,
fechas inconsistentes y observaciones sin horizonte. Solo se usan modalidad, tipo, segmento,
departamento, precio, lotes, plazo previsto y estacionalidad. No entran ganador, estado final,
visualizaciones, respuestas ni códigos de empresa/proceso.

**Partición y maduración:** entrenamiento hasta 2022-12-31; validación durante 2024; prueba desde
2025-10-01 hasta el último día con 240 días observables. Las etiquetas del entrenamiento ya
maduraron al empezar validación; las de validación, antes de prueba. Se reportan los intervalos
omitidos. Se aprende imputación, recorte y categorías exclusivamente en entrenamiento.

Se implementa una regresión logística L2 pequeña en NumPy (no requiere instalaciones):
gradiente con paso basado en cota de Lipschitz, comprobación de convergencia y ajuste de L2
por pérdida logarítmica de validación. Se compara con una probabilidad constante entrenada.
Se reportan Brier, log-loss, AUC y AP, prevalencia y calibración; no se usa exactitud como único criterio.
También se compara con la prevalencia más reciente de validación, observable antes de prueba.
Se usa fecha de firma como tiempo de evento; no equivale necesariamente al momento en que el registro
estuvo publicado. Deben auditarse retrasos de reporte para una validación histórica operacional.

### A15 · Modelo auxiliar y evaluación fuera del tiempo de entrenamiento

**Por qué se hace.** Se comprueba si los atributos seleccionados añaden valor predictivo sobre una referencia sencilla, antes de plantear algoritmos más complejos.

**Qué se hace.** Se ajusta una regresión logística regularizada para estimar evidencia de contrato a 240 días. Se compara con asignar a todos la misma probabilidad, calculada en entrenamiento o en validación reciente.

**Cómo se hace.** Entrenamiento termina en 2022, validación usa 2024 y prueba comienza en octubre de 2025. Los intervalos permiten que las etiquetas anteriores maduren. Medianas, recortes y vocabulario categórico se aprenden solo en entrenamiento. La regularización L2 reduce coeficientes extremos; se escoge entre tres intensidades usando log-loss de validación. Brier es el error cuadrático de probabilidades y log-loss penaliza especialmente la confianza equivocada: en ambos, menor es mejor. AUC mide ordenación; AP resume recuperación de positivos y debe compararse con su prevalencia.

In [ ]:
train=observed.loc[observed.fecha_de_publicacion_del < '2023-01-01'].copy()
valid=observed.loc[observed.fecha_de_publicacion_del.between('2024-01-01','2024-12-31')].copy()
test=observed.loc[observed.fecha_de_publicacion_del >= '2025-10-01'].copy()
split_info=pd.DataFrame([{'particion':name,'n':len(z),'inicio':z.fecha_de_publicacion_del.min(),
                         'fin':z.fecha_de_publicacion_del.max(),'prevalencia':z.contrato_240d.mean()}
                        for name,z in [('entrenamiento',train),('validacion',valid),('prueba',test)]])
display(split_info)
say(f'Quedan {len(observed)-len(train)-len(valid)-len(test):,} observaciones fuera de estas particiones por diseño temporal. '
    'No se interpreta un cero como fracaso demostrado. La fotografía final de atributos sigue siendo una limitación temporal.')
assert min(len(train),len(valid),len(test))>=100, 'No hay soporte suficiente para el experimento temporal.'
cats=['modalidad_de_contratacion','tipo_de_contrato','segmento','departamento_entidad']
def num_features(z):
    out=pd.DataFrame(index=z.index)
    out['log_precio']=np.log1p(z.precio_base.where(z.precio_base>0))
    out['log_lotes']=np.log1p(z.numero_de_lotes.where(z.numero_de_lotes>=0))
    out['log_plazo']=np.log1p(z.plazo_bruto.where(z.plazo_bruto>=0))
    month=z.fecha_de_publicacion_del.dt.month
    out['mes_sin']=np.sin(2*np.pi*month/12); out['mes_cos']=np.cos(2*np.pi*month/12)
    return out.replace([np.inf,-np.inf],np.nan)
nt=num_features(train)
medians=nt.median().fillna(0); low=nt.quantile(.01).fillna(0); high=nt.quantile(.99).fillna(0)
centers=nt.clip(low,high,axis=1).fillna(medians).mean()
scales=nt.clip(low,high,axis=1).fillna(medians).std().replace(0,1).fillna(1)
levels={}
for col in cats:
    counts_train=train[col].fillna('SIN_DATO').value_counts()
    levels[col]=counts_train[counts_train>=100].head(12).index.tolist()
display(pd.Series({col:values[0] for col,values in levels.items()},name='categoría de referencia'))
def design(z):
    n=num_features(z); missing=n.isna().astype(float).add_suffix('_faltante')
    n=(n.clip(low,high,axis=1).fillna(medians)-centers)/scales
    out=pd.concat([n,missing],axis=1)
    for col in cats:
        a=z[col].fillna('SIN_DATO').astype(str)
        # La categoría más frecuente es la referencia; vocabulario fijado en train.
        for level in levels[col][1:]: out[f'{col}={level}']=a.eq(level).astype(float)
        out[f'{col}=OTRAS']=(~a.isin(levels[col])).astype(float)
    out.insert(0,'intercepto',1.)
    return out
Xt,Xv,Xe=design(train),design(valid),design(test)
yt,yv,ye=[z.contrato_240d.to_numpy(float) for z in [train,valid,test]]
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-35,35)))
def logistic(X,y,lam,max_iter=1600,tol=1e-6):
    X=np.asarray(X,float); n=X.shape[0]; w=np.zeros(X.shape[1])
    penalty=np.ones(len(w)); penalty[0]=0
    L=np.linalg.eigvalsh(X.T@X/n).max()/4+lam
    step=1/max(L,1e-6)
    for iteration in range(max_iter):
        grad=X.T@(sigmoid(X@w)-y)/n+lam*w*penalty
        w-=step*grad
        if np.max(np.abs(grad))<tol: break
    return w,{'iteraciones':iteration+1,'grad_max':float(np.max(np.abs(grad))),'convergio':bool(np.max(np.abs(grad))<tol)}
def metrics(y,p):
    y=np.asarray(y,float); p=np.clip(np.asarray(p,float),1e-9,1-1e-9)
    pos=y.sum(); neg=len(y)-pos
    auc=(pd.Series(p).rank().to_numpy()[y==1].sum()-pos*(pos+1)/2)/(pos*neg) if pos and neg else np.nan
    tmp=pd.DataFrame({'p':p,'y':y}).groupby('p').y.agg(['sum','count']).sort_index(ascending=False)
    tp=tmp['sum'].cumsum(); seen=tmp['count'].cumsum()
    ap=float(((tp/seen)*(tmp['sum']/pos)).sum()) if pos else np.nan
    return {'n':len(y),'prevalencia':y.mean(),'Brier':np.mean((p-y)**2),
            'log_loss':-np.mean(y*np.log(p)+(1-y)*np.log(1-p)),'ROC_AUC':auc,'AP':ap}
fits=[]
for lam in [.01,.1,1.]:
    w,diag=logistic(Xt,yt,lam)
    fits.append({'lambda':lam,'w':w,**diag,'logloss_valid':metrics(yv,sigmoid(Xv.to_numpy()@w))['log_loss']})
best=min(fits,key=lambda d:d['logloss_valid'])
display(pd.DataFrame([{k:v for k,v in d.items() if k!='w'} for d in fits]))
pred=sigmoid(Xe.to_numpy()@best['w'])
model_results=pd.DataFrame([{'modelo':'Constante train',**metrics(ye,np.repeat(yt.mean(),len(ye)))},
                           {'modelo':'Constante validación',**metrics(ye,np.repeat(yv.mean(),len(ye)))},
                           {'modelo':'Logística auxiliar',**metrics(ye,pred)}])
display(model_results)
say(f"L2 seleccionado con validación: {best['lambda']}; convergencia: {best['convergio']}. "
    'Si no converge, las métricas son provisionales y debe ajustarse el optimizador antes de interpretar coeficientes.')

**Conclusión del análisis A15**

Se usaron **14.900 casos de entrenamiento, 4.273 de validación y 1.595 de prueba**. La prevalencia sube de **76,1%** a **82,7%** y **83,8%**, respectivamente: existe cambio temporal en la tasa base.

La logística converge, pero su **Brier 0,1425** es peor que el **0,1361** de la constante reciente; también pierde en log-loss (**0,4621 frente a 0,4440**). Su AUC es **0,559**, que indica ordenación débil. AP de 0,858 debe leerse frente a una prevalencia ya alta de 0,838, no como 85,8% de aciertos. Con esta evidencia no se justifica incorporar el modelo auxiliar al ranking; habría que revisar etiquetas, cobertura y estabilidad temporal antes de aumentar complejidad.

### A16 · Calibración, coeficientes y robustez del modelo auxiliar

**Por qué se hace.** Una puntuación puede ordenar algo mejor que el azar y aun así asignar probabilidades equivocadas. También se necesita comprobar si la diferencia con el baseline es estable dentro de la muestra.

**Qué se hace.** Se confrontan probabilidades predichas con frecuencias observadas, se muestran los coeficientes de mayor magnitud y se estima incertidumbre del cambio de Brier.

**Cómo se hace.** Las predicciones se agrupan en diez cuantiles; cada punto de calibración compara probabilidad media y tasa real. La diagonal representa coincidencia perfecta. Los coeficientes numéricos corresponden a variables transformadas/estandarizadas y las categorías a la referencia indicada en la celda anterior. Para el bootstrap se remuestrean entidades completas 300 veces, manteniendo juntos sus procesos, y se calcula `Brier logística − Brier constante de validación`.

In [ ]:
cal=pd.DataFrame({'p':pred,'y':ye})
cal['grupo']=pd.qcut(cal.p,10,duplicates='drop')
calibration=cal.groupby('grupo',observed=True).agg(p=('p','mean'),tasa=('y','mean'),n=('y','size'))
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot([0,1],[0,1],'--',color='gray'); ax[0].plot(calibration.p,calibration.tasa,'o-',label='Prueba futura')
ax[0].set(title='Calibración fuera de tiempo',xlabel='Probabilidad auxiliar media',ylabel='Frecuencia de contrato ≤240d',xlim=(0,1),ylim=(0,1))
coef=pd.Series(best['w'][1:],index=Xt.columns[1:]); chosen=coef.abs().nlargest(12).index
coef_plot=coef.loc[chosen].sort_values()
coef_plot.index=[s.replace('modalidad_de_contratacion=','Modalidad: ').replace('tipo_de_contrato=','Tipo: ').replace('departamento_entidad=','Depto: ').replace('segmento=','Segmento: ') for s in coef_plot.index]
coef_plot.plot.barh(ax=ax[1],color='#246B8E')
ax[1].set(title='Coeficientes L2: asociación condicionada',xlabel='Log-odds; numéricas estandarizadas; no causal')
plt.tight_layout(); plt.show(); display(calibration)
# Incertidumbre de mejora de Brier: remuestreo de entidades completas.
err=(pred-ye)**2-(yv.mean()-ye)**2
errdf=pd.DataFrame({'entidad':test.nit_entidad.fillna('SIN_ENTIDAD').to_numpy(),'delta':err})
agg=errdf.groupby('entidad').delta.agg(['sum','count'])
boot=[]; rr=np.random.default_rng(SEED)
for _ in range(300):
    draw=rr.integers(0,len(agg),len(agg)); b=agg.iloc[draw]; boot.append(b['sum'].sum()/b['count'].sum())
ci=np.quantile(boot,[.025,.975])
say(f'**H6:** ΔBrier (logística − constante de validación) = {err.mean():.4f}; IC bootstrap por entidad '
    f'95% [{ci[0]:.4f}, {ci[1]:.4f}]. Negativo favorece a la logística. '
    'El intervalo describe este bloque temporal; no cubre cambios futuros ni errores de etiqueta.')
say('**Decisión del piloto:** ' + ('la logística mejora el Brier en este bloque, sujeto a sus límites de cobertura.'
    if err.mean()<0 else 'la logística no mejora el Brier del baseline reciente; no se justifica incorporarla al ranking con esta evidencia.'))

**Conclusión del análisis A16**

Los puntos de calibración quedan por encima de la diagonal: el modelo tiende a **subestimar la frecuencia de contratos** en la prueba. Por ejemplo, el grupo con predicción media 0,641 muestra una frecuencia cercana a 0,756.

El cambio de Brier es **+0,0064**, con intervalo bootstrap de 95% **[0,0023; 0,0100]**. El signo positivo indica mayor error que la referencia reciente; esta evaluación no respalda H6. Los coeficientes describen asociaciones condicionadas por la muestra y la regularización, no efectos causales. El bootstrap no incorpora todos los errores de etiquetas ni cambios futuros y no convierte este bloque temporal en validación externa.

## 8. Ranking empresa–oportunidad: backtest de señales implícitas

**Ventana predefinida:** historial de contratos firmados desde 2017 hasta 2024-12-31;
candidatos publicados en enero de 2025, de cualquier estado actual dentro del universo;
positivo observable = la empresa obtuvo un contrato vinculado dentro de 240 días de publicación.

Se seleccionan proveedores con hash reproducible (1/4) y al menos tres contratos históricos válidos.
Se limita a los primeros 1.000 por hash **sin mirar sus resultados futuros**. Esto evalúa empresas
con historial; se reporta cuántas no tienen interacciones futuras observadas y quedan fuera de Recall.

Este es un **ranking retrospectivo de publicaciones del mes**, evaluado al reunirse el lote. No simula
que las ofertas de finales de enero ya existían el 1 de enero, ni reproduce qué seguía abierto cada día.
Una evaluación online requiere instantáneas diarias, ventanas de recepción y candidatos disponibles a cada instante.

Puntaje fijo propuesto: 70% afinidad de segmento + 20% cercanía de escala monetaria + 10% historial con la entidad.
Los pesos no se ajustan con enero. Comparadores: popularidad de segmento, afinidad sola y ablación sin entidad.
La escala de contratos anteriores es una aproximación a experiencia, no capacidad financiera certificada.

### A17 · Construcción del experimento empresa–oportunidad

**Por qué se hace.** Para recomendar hay que evaluar pares empresa–oferta: una tasa general de contratación del proceso no expresa utilidad para una empresa concreta.

**Qué se hace.** Se forman perfiles con historia previa, un conjunto de ofertas publicadas posteriormente y positivos que indican contratos futuros observados para esas empresas.

**Cómo se hace.** El historial utiliza firmas anteriores a enero de 2025. Se selecciona reproduciblemente un cuarto de los códigos de proveedor y hasta 1.000 empresas con al menos tres contratos históricos válidos, sin mirar resultados futuros. Los candidatos son publicaciones de enero de 2025; se excluyen enlaces ambiguos, categoría ausente e identificadores repetidos en el mes. Un positivo es un contrato enlazado dentro de los siguientes 240 días. El experimento ordena retrospectivamente el lote mensual; no reproduce una lista diaria de ofertas abiertas.

In [ ]:
history=query(f"""SELECT id_contrato,codigo_proveedor,nit_entidad,valor_del_contrato,
 fecha_de_firma,left(substring(codigo_de_categoria_principal from '([0-9]{{8}})$'),2) AS segmento
 FROM secop.contratos WHERE {SCOPE} AND fecha_de_firma>='2017-01-01' AND fecha_de_firma<'2025-01-01'
 AND codigo_proveedor IS NOT NULL AND left(md5(codigo_proveedor),1) IN ('0','1','2','3')""")
duplicate_history=int(history.id_contrato.duplicated().sum())
history=history.sort_values(['fecha_de_firma','id_contrato']).drop_duplicates('id_contrato')
history['valor_del_contrato']=pd.to_numeric(history.valor_del_contrato,errors='coerce').replace([np.inf,-np.inf],np.nan)
history=history.loc[history.segmento.notna()].copy()
counts=history.groupby('codigo_proveedor').size()
import hashlib
companies=sorted(counts[counts>=3].index,key=lambda s:hashlib.md5(str(s).encode()).hexdigest())[:1000]
hist=history.loc[history.codigo_proveedor.isin(companies)].copy()
candidate=query(f"""SELECT DISTINCT ON(id_del_proceso) id_del_proceso,id_del_portafolio,
 nit_entidad,precio_base,fecha_de_publicacion_del,fecha_de_recepcion_de,
 modalidad_de_contratacion,departamento_entidad,
 left(substring(codigo_principal_de_categoria from '([0-9]{{8}})$'),2) AS segmento,
 count(*) OVER(PARTITION BY id_del_proceso) AS filas_del_id
 FROM secop.procesos WHERE {SCOPE} AND fecha_de_publicacion_del>='2025-01-01'
 AND fecha_de_publicacion_del<'2025-02-01' AND id_del_proceso IS NOT NULL
 ORDER BY id_del_proceso,fecha_de_publicacion_del,id_del_portafolio NULLS LAST""")
candidate=candidate.merge(portfolio_counts,on='id_del_portafolio',how='left',validate='many_to_one')
candidate_all_n=len(candidate)
candidate=candidate.loc[candidate.n_procesos_portafolio.eq(1)&candidate.segmento.notna()&candidate.filas_del_id.eq(1)].copy()
candidate=candidate.sort_values('id_del_proceso').reset_index(drop=True)
future=query(f"""SELECT codigo_proveedor,proceso_de_compra,fecha_de_firma,id_contrato
 FROM secop.contratos WHERE fecha_de_firma>='2025-01-01' AND fecha_de_firma<'2025-10-01'
 AND codigo_proveedor IS NOT NULL AND left(md5(codigo_proveedor),1) IN ('0','1','2','3')""")
positive=future.loc[future.codigo_proveedor.isin(companies)].merge(
 candidate[['id_del_proceso','id_del_portafolio','fecha_de_publicacion_del']],
 left_on='proceso_de_compra',right_on='id_del_portafolio',how='inner',validate='many_to_one')
delay=(pd.to_datetime(positive.fecha_de_firma)-pd.to_datetime(positive.fecha_de_publicacion_del)).dt.total_seconds()/86400
positive=positive.loc[delay.between(0,HORIZONTE)].drop_duplicates(['codigo_proveedor','id_del_proceso'])
global_pop=query(f"""SELECT left(substring(codigo_de_categoria_principal from '([0-9]{{8}})$'),2) AS segmento,
 count(DISTINCT id_contrato) AS n FROM secop.contratos WHERE {SCOPE}
 AND fecha_de_firma>='2017-01-01' AND fecha_de_firma<'2025-01-01' GROUP BY 1""")
pop=global_pop.set_index('segmento').n
display(pd.Series({'contratos_historia_hash_1_4':len(history),'duplicados_historia_eliminados':duplicate_history,
 'empresas_warm_seleccionadas':len(companies),'candidatos_publicados':candidate_all_n,
 'candidatos_evaluables':len(candidate),'empresas_con_positivo_observado':positive.codigo_proveedor.nunique(),
 'pares_positivos_distintos':len(positive)}))
say('No se filtran candidatos por su estado final ni por adjudicación. El filtro de clave y categoría '
    'define cobertura técnica; se excluyen conservadoramente identificadores repetidos en el mes. '
    'Estos filtros pueden excluir ofertas relevantes. Se mide recuperación de contratos observados '
    'entre candidatos, no relevancia humana, tasa de participación ni probabilidad de ganar.')

**Conclusión del análisis A17**

El historial válido de la fracción seleccionada contiene **97.472 contratos**. De **3.515 procesos publicados** en la ventana quedan **2.187 candidatos evaluables**. Entre las 1.000 empresas elegidas, solo **37** tienen al menos un positivo observable, que suman **60 pares empresa–proceso**.

Esto permite un piloto de recuperación para empresas con experiencia, pero la cobertura de evaluación positiva es **3,7%**. Las otras 963 empresas no se convierten en ejemplos negativos: podrían haber tenido ofertas relevantes sin contratar. No se han evaluado empresas nuevas sin historial ni relevancia declarada por usuarios; esas son poblaciones y etiquetas que el producto necesita incorporar.

### A18 · Comparación de rankings y descubrimiento de clientes nuevos

**Por qué se hace.** Se necesita comprobar qué aporta personalizar y si las recomendaciones ayudan a encontrar compradores nuevos o principalmente a recuperar relaciones existentes.

**Qué se hace.** Se comparan popularidad de segmento, afinidad de categoría, categoría más escala monetaria y un puntaje que añade familiaridad con la entidad. Se examinan por separado positivos con compradores conocidos y nuevos.

**Cómo se hace.** El combinado suma 70% afinidad de segmento, 20% cercanía entre presupuesto y escala de contratos anteriores, y 10% historial con la entidad. Son pesos exploratorios fijos, no optimizados para esta prueba. `Recall@20` es la fracción de positivos de cada empresa presentes en sus 20 primeras recomendaciones; después se promedia entre empresas. `Hit@20` mide cuántas empresas recuperan al menos uno; NDCG da más valor a recuperarlos al principio. La cobertura es la fracción del catálogo recomendada alguna vez. Para clientes nuevos se compara el NIT comprador con el historial anterior.

In [ ]:
cat_share=pd.crosstab(hist.codigo_proveedor,hist.segmento).div(hist.groupby('codigo_proveedor').size(),axis=0)
ent_share=pd.crosstab(hist.codigo_proveedor,hist.nit_entidad).div(hist.groupby('codigo_proveedor').size(),axis=0)
median_value=hist.loc[hist.valor_del_contrato>0].groupby('codigo_proveedor').valor_del_contrato.median()
ids=candidate.id_del_proceso.to_numpy(); segments=candidate.segmento
budget=np.log1p(pd.to_numeric(candidate.precio_base,errors='coerce').where(candidate.precio_base>0)).to_numpy()
pop_score=segments.map(pop).fillna(0).to_numpy(float)
truth=positive.groupby('codigo_proveedor').id_del_proceso.agg(set).to_dict()
entity_by_id=candidate.set_index('id_del_proceso').nit_entidad.to_dict()
known_entities=hist.groupby('codigo_proveedor').nit_entidad.agg(lambda s:set(s.dropna())).to_dict()
novelty_rows=[]
K=20; ranking_rows=[]; top_sets={name:set() for name in ['Popularidad','Categoría','Sin entidad','Combinado']}
examples=[]
for company in companies:
    affinity=segments.map(cat_share.loc[company]).fillna(0).to_numpy(float)
    familiarity=candidate.nit_entidad.map(ent_share.loc[company]).fillna(0).to_numpy(float) if company in ent_share.index else np.zeros(len(candidate))
    median=median_value.get(company,np.nan)
    money=np.exp(-np.abs(budget-np.log1p(median))) if pd.notna(median) else np.zeros(len(candidate))
    money=np.nan_to_num(money,nan=0.)
    scores={'Popularidad':pop_score,'Categoría':affinity,'Sin entidad':.7*affinity+.2*money,
            'Combinado':.7*affinity+.2*money+.1*familiarity}
    relevant=truth.get(company,set())
    old_entities=known_entities.get(company,set())
    new_relevant={i for i in relevant if pd.notna(entity_by_id.get(i)) and entity_by_id.get(i) not in old_entities}
    old_relevant={i for i in relevant if entity_by_id.get(i) in old_entities}
    # Empates reproducibles por hash empresa-proceso, evitando orden de identificadores.
    tie=np.array([int(hashlib.md5((str(company)+'|'+str(i)).encode()).hexdigest()[:8],16) for i in ids],dtype=np.uint64)
    for name,score in scores.items():
        order=np.lexsort((tie,-score))[:K]; chosen=ids[order]; top_sets[name].update(chosen)
        measurable=[i for i in chosen if pd.notna(entity_by_id.get(i))]
        novelty_rows.append({'modelo':name,'novedad_entidad':np.mean([entity_by_id[i] not in old_entities for i in measurable]) if measurable else np.nan})
        if relevant:
            hit=np.isin(chosen,list(relevant)).astype(float)
            ideal=(1/np.log2(np.arange(min(K,len(relevant)))+2)).sum()
            ranking_rows.append({'empresa':company,'modelo':name,'positivos':len(relevant),
               'recall20':hit.sum()/len(relevant),'ndcg20':(hit/np.log2(np.arange(len(hit))+2)).sum()/ideal,
               'hit20':float(hit.sum()>0),
               'recall_entidad_nueva':len(set(chosen)&new_relevant)/len(new_relevant) if new_relevant else np.nan,
               'recall_entidad_conocida':len(set(chosen)&old_relevant)/len(old_relevant) if old_relevant else np.nan})
        if company==companies[0] and name=='Combinado':
            example=candidate.iloc[order[:8]].copy()
            example['afinidad_segmento']=affinity[order[:8]]; example['cercania_escala']=money[order[:8]]
            example['familiaridad_entidad']=familiarity[order[:8]]; example['puntaje']=score[order[:8]]
ranking_eval=pd.DataFrame(ranking_rows)
assert len(ranking_eval)>0, 'No hay positivos observados en esta cohorte; no se puede evaluar recuperación.'
ranking_summary=ranking_eval.groupby('modelo').agg(empresas=('empresa','size'),Recall20=('recall20','mean'),
                      NDCG20=('ndcg20','mean'),Hit20=('hit20','mean'))
ranking_summary['cobertura_catalogo_top20']=[len(top_sets[i])/len(candidate) for i in ranking_summary.index]
display(ranking_summary)
novelty=pd.DataFrame(novelty_rows).groupby('modelo').novedad_entidad.mean()
new_clients=ranking_eval.groupby('modelo').agg(empresas_con_nuevo_cliente=('recall_entidad_nueva','count'),
    recall_cliente_nuevo=('recall_entidad_nueva','mean'),empresas_con_cliente_conocido=('recall_entidad_conocida','count'),
    recall_cliente_conocido=('recall_entidad_conocida','mean'))
new_clients['fraccion_top20_entidades_nuevas']=novelty
display(new_clients)
display(example[['id_del_proceso','segmento','precio_base','afinidad_segmento','cercania_escala','familiaridad_entidad','puntaje']])
say('Ejemplo de explicación para una empresa seleccionada por hash: el puntaje no es una probabilidad. '
    'La cobertura Top20 se mide entre todas las empresas; Recall/NDCG solo entre empresas con positivos '
    'observados. No se calcula precisión de relevancia porque los demás pares no están etiquetados como irrelevantes.')
say('La tabla de clientes nuevos usa NIT de entidad tal como está almacenado; requiere homologación '
    'para uso operacional. Separa descubrimiento de recuperación de clientes conocidos. La cobertura '
    'de evaluación positiva es ' + f'{len(truth)/len(companies):.1%} de las empresas seleccionadas; '
    'las demás no son negativos de relevancia.')

**Conclusión del análisis A18**

El Recall@20 **macro** es **1,35% para popularidad, 9,01% para categoría, 22,01% sin entidad y 41,38% para el combinado**. Este último recupera al menos un positivo para **51,35%** de las 37 empresas evaluables y logra NDCG@20 de 0,298. Estos porcentajes no son probabilidades de ganar ni precisión de relevancia.

La distinción de clientes cambia la lectura: el combinado recupera **14,3%** de los positivos con entidades nuevas en solo **7 empresas**, frente a **45,7%** con entidades conocidas en **34 empresas**. Algunos proveedores aparecen en ambos grupos. El beneficio está mucho más respaldado para relaciones ya conocidas; el descubrimiento requiere más datos y evaluación propia.

El puntaje se puede explicar por categoría, escala y experiencia con el comprador. Su transparencia es útil para un piloto, pero ni el puntaje ni la escala histórica certifican capacidad financiera o cumplimiento de requisitos.

### A19 · Comparaciones pareadas e incertidumbre del ranking

**Por qué se hace.** Con pocas empresas, una mejora media puede depender de algunos casos. Es necesario comparar métodos sobre las mismas empresas y mostrar la variabilidad de la diferencia.

**Qué se hace.** Se estiman diferencias de Recall@20 entre rankings y se muestra el alcance de las ofertas recomendadas.

**Cómo se hace.** Se construye una fila por empresa con todos sus resultados. Se remuestrean empresas completas 500 veces y, en cada repetición, se calcula la diferencia media entre los mismos métodos. Los percentiles 2,5 y 97,5 forman el intervalo descriptivo de 95%. Mantener la comparación pareada evita confundir diferencias de población con diferencias de método.

In [ ]:
paired=ranking_eval.pivot(index='empresa',columns='modelo',values='recall20')
comparisons=[]
for chosen_model,alt in [('Categoría','Popularidad'),('Combinado','Popularidad'),('Combinado','Categoría'),('Combinado','Sin entidad')]:
    delta=(paired[chosen_model]-paired[alt]).dropna().to_numpy()
    rr=np.random.default_rng(SEED)
    means=np.array([rr.choice(delta,len(delta),replace=True).mean() for _ in range(500)])
    lo,hi=np.quantile(means,[.025,.975])
    comparisons.append({'comparación':chosen_model+' − '+alt,'delta_Recall20':delta.mean(),'IC95_lo':lo,'IC95_hi':hi,'empresas':len(delta)})
ranking_ci=pd.DataFrame(comparisons); display(ranking_ci)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ranking_summary.Recall20.sort_values().plot.barh(ax=ax[0],color='#246B8E'); ax[0].set(title='Recuperación de interacciones positivas',xlabel='Recall@20 macro por empresa')
ranking_summary.cobertura_catalogo_top20.sort_values().plot.barh(ax=ax[1],color='#3A917B'); ax[1].set(title='Diversidad de exposición',xlabel='Fracción de candidatos recomendada al menos una vez')
plt.tight_layout(); plt.show()
say('Los intervalos se obtienen remuestreando empresas, manteniendo pares los métodos. '
    'Empresas pueden compartir procesos: estos intervalos no recogen toda la dependencia del mercado. '
    'Una ganancia en recuperar contratos históricos puede reforzar incumbencia; se necesita juicio humano sobre encaje y novedad.')

**Conclusión del análisis A19**

Categoría mejora frente a popularidad en **7,66 puntos porcentuales**, con intervalo aproximado **[0,66; 16,22] puntos**: H1 recibe respaldo exploratorio en esta cohorte. El combinado mejora frente a popularidad en **40,03 puntos** y, al añadir familiaridad con la entidad, mejora sobre la versión sin entidad en **19,37 puntos**, con intervalo **[8,11; 32,67]**. H5 tiene respaldo para recuperación histórica, no para un efecto causal de conocer al comprador.

Los cuatro intervalos reportados son positivos, pero corresponden a un solo mes y 37 empresas. Puede haber dependencia porque varias empresas comparten oportunidades; tampoco son intervalos simultáneos ajustados por múltiples contrastes. Los resultados justifican continuar el piloto y repetir ventanas, no declarar superioridad general ni resolver la validación de relevancia.

## 9. Qué modelos explorar después y cómo evaluar las hipótesis

| Modelo / análisis | Qué pregunta resuelve | Entradas y objetivo | Validación y riesgo |
|---|---|---|---|
| Recuperación por reglas + contenido | H1: perfil vs oferta, incluyendo empresas nuevas | UNSPSC jerárquico, texto del objeto, capacidades declaradas, ubicación aceptada | relevancia evaluada por empresas; Recall/NDCG con juicios; no garantiza habilitación |
| Ranking aprendido con feedback | priorizar ofertas útiles | pares empresa–oferta; clic, guardado, participación y motivo de descarte | cortes temporales, evaluación por empresa, corrección de exposición; datos hoy ausentes |
| Ranking implícito BPR/ALS | H1/H5: historial y afinidad | contratos históricos como positivos; pares no observados muestreados | no llamarlos pérdidas; sesgo de ganadores, popularidad y cold-start; backtests repetidos |
| Logística/GAM con interacciones | H3/H6: heterogeneidad de conversión del proceso | información al publicar, contrato observado a horizonte | calibración temporal; splines precio/plazo y modalidad; incertidumbre por entidad |
| Boosting con categorías | relaciones no lineales del auxiliar | misma lista permitida, nunca fuentes del objetivo | ganancia sobre baseline temporal; SHAP describe asociación, no causa |
| Supervivencia / riesgos competitivos | cuánto tarda y qué desenlace ocurre | fechas/eventos fiables y censura; firma vs cancelación | se necesitan fechas de cancelación e historial; «sin contrato» no es evento negativo |
| Redes proveedor–entidad y grupos | H4/H5: concentración, comunidades y descubrimiento | aristas anteriores a predicción, membresías fechadas | evitar aristas futuras; separar efecto tamaño y antigüedad; auditar exposición |

Para H2 usar segmentación y, con etiquetas de participación, un modelo con interacción
territorio × categoría × escala. Para H4 estimar concentración también por valor y familia UNSPSC,
con auditoría de montos y duplicados, antes de explicar diferencias por «competencia».

**Diseño de la iniciativa:** recolectar perfil declarado (categorías, experiencia, regiones, tamaño de contratos,
requisitos que puede cumplir); obtener oportunidades vigentes y sus documentos; aplicar filtros habilitantes;
recuperar candidatos por similitud; ordenar con razones comprensibles; registrar evaluación humana.
Las bases actuales no contienen el objeto detallado, pliegos ni todos los requisitos necesarios para esa decisión.

### A20 · Síntesis de hipótesis y elección del siguiente paso

**Por qué se hace.** El EDA debe terminar en decisiones sobre datos, objetivos y evaluación; acumular gráficos o algoritmos no define qué construir.

**Qué se hace.** Se reúne la evidencia de las seis hipótesis y se distingue lo ejecutado de los modelos propuestos para una siguiente fase.

**Cómo se hace.** Se relaciona cada resultado con su población, señal observada y limitación. Los modelos futuros de la tabla anterior se eligen por la pregunta: contenido y reglas para encaje; ranking con feedback para utilidad; modelos implícitos para historia; GAM/boosting para relaciones del proceso; supervivencia para tiempos; redes para estructura de vínculos. Solo la logística y los rankings descritos han sido evaluados aquí. Las demás alternativas aún no tienen resultados experimentales.

In [ ]:
h1=ranking_ci.loc[ranking_ci['comparación'].eq('Categoría − Popularidad')].iloc[0]
h5=ranking_ci.loc[ranking_ci['comparación'].eq('Combinado − Sin entidad')].iloc[0]
decisions=pd.DataFrame([
 {'hipótesis':'H1 · afinidad vs popularidad','evidencia':f"ΔRecall20={h1.delta_Recall20:.4f}; IC [{h1.IC95_lo:.4f}, {h1.IC95_hi:.4f}]",'lectura':'Piloto de recuperación; repetir meses y ampliar señales de relevancia'},
 {'hipótesis':'H2 · territorio por segmento','evidencia':f'{geo.comparable.mean():.1%} con territorios comparables','lectura':'Descriptivo, catálogo actual; no efecto causal'},
 {'hipótesis':'H3 · mezcla de modalidades','evidencia':f'{len(reversals)} modalidades con signo opuesto al agregado','lectura':'Investigar interacción; no fijar una regla global de presupuesto'},
 {'hipótesis':'H4 · concentración','evidencia':f'{len(concentration)} segmentos con al menos 100 contratos','lectura':'HHI por conteo, no por valor; revisar definiciones y barreras observables'},
 {'hipótesis':'H5 · familiaridad con entidad','evidencia':f"ΔRecall20={h5.delta_Recall20:.4f}; IC [{h5.IC95_lo:.4f}, {h5.IC95_hi:.4f}]",'lectura':'Comparar recuperación con descubrimiento y sesgo hacia incumbentes'},
 {'hipótesis':'H6 · auxiliar temporal','evidencia':f'ΔBrier={err.mean():.4f}; IC [{ci[0]:.4f}, {ci[1]:.4f}]','lectura':'Evidencia de contrato, nunca probabilidad de ganar de una empresa'}
])
display(decisions.style.hide(axis='index'))
say(f'**Lectura del ranking:** se evalúan {len(truth)} de {len(companies)} empresas, con '
    f'{len(positive)} interacciones positivas y {len(candidate):,} candidatos. '
    f'El combinado obtiene Recall@20={ranking_summary.loc["Combinado","Recall20"]:.1%}, '
    f'frente a {ranking_summary.loc["Popularidad","Recall20"]:.1%} de popularidad. '
    'Este resultado respalda continuar el piloto de recuperación, no lanzar una estimación de éxito comercial.')
novel_result=new_clients.loc['Combinado']
say(f'**Descubrimiento:** para el combinado, Recall@20 con entidades nuevas = '
    f'{novel_result.recall_cliente_nuevo:.1%} en {int(novel_result.empresas_con_nuevo_cliente)} empresas '
    f'con ese tipo de positivo; con entidades conocidas = {novel_result.recall_cliente_conocido:.1%}. '
    'La comparación tiene poblaciones distintas; sirve para detectar dependencia del historial y orientar '
    'evaluaciones separadas, no para atribuir causalidad a conocer al comprador.')
say('**Objetivo recomendado para el producto:** relevancia de una oferta para una empresa, '
    'validada por esa empresa y condicionada a requisitos. Mientras se recogen esas etiquetas, '
    'el ranking histórico es un baseline de recuperación explicable. El objetivo heredado de Gold '
    'y el auxiliar a 240 días son señales del proceso, no sustitutos de relevancia individual.')
say('**Próxima validación:** repetir al menos seis ventanas mensuales con perfiles calculados antes de cada '
    'ventana; evaluar nuevos proveedores por separado; sensibilidad a segmento/familia, K=5/10/20 y '
    'horizonte 180/240/365; inspección humana de recomendaciones y catálogo de requisitos. '
    'El notebook no registra decisiones definitivas a partir de un solo mes.')
engine.dispose()

**Conclusión del análisis A20**

La primera versión de la iniciativa debería **ordenar oportunidades por afinidad explicable con el perfil empresarial** y pedir a las empresas que validen su relevancia y requisitos. Los contratos históricos sirven como señal positiva inicial, pero no reemplazan esas etiquetas.

| Hipótesis | Conclusión de esta ejecución | Decisión que permite |
|---|---|---|
| H1 · afinidad de categoría | Mejora sobre popularidad en la pequeña cohorte evaluable. | Mantener como baseline y contrastar familia/clase UNSPSC en otros meses. |
| H2 · territorio | La coincidencia de departamento varía mucho entre segmentos. | Ofrecer preferencias geográficas configurables; no imponer coincidencia universal. |
| H3 · mezcla de modalidades | Se atenúa la asociación precio–resultado al agregar, sin inversión de signo observada. | Explorar interacciones y corregir la cobertura de portafolios antes de generalizar. |
| H4 · concentración | Existen segmentos con muchos proveedores nominales y pocos efectivos. | Estudiar especialización e identidad empresarial; no inferir barreras o irregularidad. |
| H5 · familiaridad con entidad | Aumenta recuperación, con evidencia más fuerte para compradores conocidos. | Evaluar por separado descubrimiento, diversidad y empresas nuevas. |
| H6 · modelo auxiliar | La logística no supera la constante reciente en Brier/log-loss. | No incorporarla al ranking con la evidencia actual. |

La prioridad de datos es reconstruir la oportunidad a través de sus etapas, disponer de versiones históricas y requisitos, y recoger clics, guardados, motivos de descarte y decisiones de participación. Después se deben repetir ventanas temporales y validar recomendaciones con empresas reales. No se recomienda todavía un clasificador de «probabilidad de ganar»: no se conoce el conjunto completo de participantes ni de derrotas.